# 08 - Consumo: dashboard Streamlit e dados do Marketplace

**Objetivo:**
1. Publicar um **dashboard** para o time de marketing, sem sair do Snowflake.
2. Enriquecer nossos dados com um dataset **gratuito do Marketplace**.

```
 Marketplace (calendario) --join por data--> GOLD --> Dashboard Streamlit
```

**Tempo estimado:** 15 minutos

In [ ]:
%%sql -r dataframe_1
-- Contexto da sessao: quem sou eu, qual motor e qual gaveta vou usar.
-- Rode esta celula SEMPRE que abrir o notebook.
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE ESPM_WH;
USE DATABASE ESPM_HOL;
USE SCHEMA APPS;

## Parte A - Streamlit: dashboard a partir do Git

**Streamlit** transforma um script Python em um app web. O app já está no repositório, em `streamlit/app.py`.

In [ ]:
%%sql -r dataframe_2
ALTER GIT REPOSITORY ESPM_HOL.PUBLIC.HOL_REPO FETCH;

CREATE OR REPLACE STREAMLIT ESPM_HOL.APPS.DASH_LOJA_PAULISTA
  FROM '@ESPM_HOL.PUBLIC.HOL_REPO/branches/main/streamlit'
  MAIN_FILE = 'app.py'
  QUERY_WAREHOUSE = ESPM_WH
  TITLE = 'Dashboard Loja Paulista'
  COMMENT = 'ESPM_HOL';

SHOW STREAMLITS IN SCHEMA ESPM_HOL.APPS;

**Para abrir:** menu **Projects > Streamlit** > `Dashboard Loja Paulista`.

### Desafio com Cortex Code
No editor do app (botão **Edit**), peça ao Cortex Code:
> *Adicione um gráfico de barras com a receita por categoria de produto usando a tabela ESPM_HOL.GOLD_DBT.RECEITA_POR_CATEGORIA*

## Parte B - Marketplace: dados de terceiros sem copiar nada

O **Marketplace** é uma loja de dados. Quando você "pega" um dataset, ele aparece como um database na sua conta, mas os dados **não são copiados**: ficam com o fornecedor e são sempre atualizados por ele.

**Passo a passo (cliques):**
1. Menu **Data Products > Marketplace**.
2. Busque: `Calendars for Finance and Analytics` (gratuito).
3. Clique em **Get**, em *Options* nomeie o database como `ESPM_MKT_CALENDARIO` e confirme.

In [ ]:
%%sql -r dataframe_3
-- Quais tabelas vieram no dataset?
SELECT table_schema, table_name, row_count
FROM ESPM_MKT_CALENDARIO.INFORMATION_SCHEMA.TABLES
WHERE table_schema <> 'INFORMATION_SCHEMA'
ORDER BY table_schema, table_name;

**Agora é com você e o Cortex Code:** escolha uma das tabelas listadas acima e peça:
> *Faça um join entre ESPM_HOL.GOLD.RECEITA_DIARIA_CANAL e a tabela <TABELA> do database ESPM_MKT_CALENDARIO pela data, e mostre a receita média em dias úteis vs fins de semana/feriados*

Pergunta de negócio: **vale a pena investir mais em mídia perto de feriados?**

### EXERCICIO 1 - Crie o KPI que você colocaria no topo do dashboard: receita por canal de venda nos últimos 30 dias de dados

Complete os `???` na célula abaixo.

**Dica:** a coluna de dinheiro da Gold se chama `receita`; o período é de 30 dias.

**Travou?** Abra o Cortex Code e peça:
> Some a receita por canal_venda dos últimos 30 dias de dados da tabela ESPM_HOL.GOLD.RECEITA_DIARIA_CANAL

In [ ]:
%%sql -r dataframe_4
SELECT canal_venda, SUM(???) AS receita_30d, SUM(pedidos) AS pedidos_30d
FROM ESPM_HOL.GOLD.RECEITA_DIARIA_CANAL
WHERE data_pedido >= (SELECT DATEADD('day', -???, MAX(data_pedido)) FROM ESPM_HOL.GOLD.RECEITA_DIARIA_CANAL)
GROUP BY canal_venda
ORDER BY receita_30d DESC;

## Encerramento: o caminho completo

```
 CSV no GitHub --> BRONZE --> SILVER --> GOLD --> Streamlit
                                |         ^  ^
                                v         |  |
                         Snowpark e dbt --+  Marketplace
```

## Depois da aula: desligue os motores

In [ ]:
%%sql -r dataframe_5
-- Para nao consumir creditos da trial
ALTER WAREHOUSE ESPM_WH SUSPEND;
SHOW COMPUTE POOLS LIKE 'ESPM_NOTEBOOK_POOL';
-- O pool desliga sozinho 5h depois que o notebook ficar ocioso.
-- Para desligar agora: menu "Connected" do notebook > Suspend service.